# **Identifying Argument Structure in John Searle's Texts Using NLI**
---
Authored by: Sangmin Seo

This notebook runs the full pipeline from `main.py` as a Google Colab notebook with GPU support. It reuses the modularized Python files from the [NLIstruct repository](https://github.com/seo-sangmin/NLIstruct).

## 1. Introduction
Identifying the structure of an argument within a text is a challenging yet crucial task across various fields, including philosophy. Given that an argument is an expression of inferences, the remarkable benchmark accuracy achieved in machine-based Natural Language Inference (NLI) tasks—thanks to the rapid development of Large Language Models (LLMs)—is noteworthy. However, despite the similarities between natural language argument and NLI, there is a lack of research that links the two fields.

### Research Question and Claims
In this Python notebook, the central question posed is: **Can NLI identify the argument structure in 'Searle (1980)' and 'Searle (2001)' in Wilson et al. (Eds.) (2001)?** I demonstrate that **even a model with a high benchmark score fails to label the argument types in the texts correctly.** Furthermore, even if we assume the existence of a model that labels correctly, **additional techniques and justifications are needed to reliably identify argument structures in the texts.** I will argue the points in Section 4 based on the results of the other sections.

### Data
To support my claims, I will use natural language texts **'Searle (1980)'** and **'Searle (2001)'**, both authored by John Searle. For the dataset, I will also employ the **SNLI corpus** to fine-tune BERT for natural language inference.

The choice of these texts may intrigue some readers, especially given that the Chinese Room argument presented in both works contends that programs do not understand natural language. However, the aim of this study is not to defend or critique the assertions made in these texts. Instead, the works will serve as test cases to evaluate NLI's capability to identify their argumentative structures.

### Techniques
Before diving into the analysis, I will examine the two texts to understand the situational contexts of their arguments. This preparatory work will facilitate a more informed evaluation of **NLI's** capabilities in argument analysis. To prepare the texts for analysis, I will undertake several tasks, including **pre-processing, exploratory analysis, topic modeling, sentence embedding, and text summarization**.

The **fine-tuning** of the model on the SNLI corpus, discussed in Section 3.1, takes approximately 25–50 minutes when using the T4 GPU on Google Colab. In Section 3.2, I will employ a different, pre-trained model for NLI tasks, so Section 3.1 is **optional** and can be skipped.

## Setup

**This notebook requires a GPU runtime.** In Colab, go to **Runtime → Change runtime type → T4 GPU → Save** before running the cells below.

Setup steps (run in order):

1. **Pin Python to 3.10** — Colab's default Python is 3.12, but several pinned packages in `requirements.txt` (numpy 1.23.5, scipy 1.10.1, torch 2.0.1, pandas 2.0.3) ship no cp312 wheels. The first cell installs Miniforge3 24.3.0-0, which bundles Python 3.10.14, and **automatically restarts the kernel**. After the restart, continue with the remaining cells.
2. **Verify Python 3.10** is now the active interpreter.
3. **Clone the `NLIstruct` repository** so the supporting modules are importable.
4. **Install dependencies** from `requirements.txt`.
5. **Confirm CUDA** is available.
6. **Pre-download NLTK corpora** to surface any network issues before the pipeline starts.

In [ ]:
# Pin Python to 3.10 to match the original notebook environment (2023-09-22).
# Miniforge3 24.3.0-0 bundles Python 3.10.14. condacolab.install_from_url
# installs it to /usr/local and triggers an automatic kernel restart —
# the SystemExit traceback / 'Your session crashed' banner is expected.
# After the restart, run the next cell.
!pip install -q condacolab
import condacolab
condacolab.install_from_url(
    "https://github.com/conda-forge/miniforge/releases/download/24.3.0-0/Miniforge3-24.3.0-0-Linux-x86_64.sh"
)

In [ ]:
# Verify the kernel restarted into Python 3.10. Expected: 3.10.14.
import sys
print("Python:", sys.version)
assert sys.version_info[:2] == (3, 10), (
    f"Expected Python 3.10, got {sys.version_info.major}.{sys.version_info.minor}. "
    "Re-run the previous cell and wait for the kernel restart."
)
import condacolab
condacolab.check()

In [ ]:
# Clone the NLIstruct repo so the supporting modules are importable.
# Guarded so re-running the cell after a reconnect doesn't error.
![ -d NLIstruct ] || git clone https://github.com/seo-sangmin/NLIstruct.git
%cd NLIstruct

In [ ]:
# Install dependencies pinned in requirements.txt.
!pip install -q -r requirements.txt

In [ ]:
# GPU sanity check. Expected: Tesla T4 in nvidia-smi and CUDA available: True.
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. The pipeline will fall back to CPU and run very slowly.")
    print("Select Runtime → Change runtime type → T4 GPU and rerun.")

In [ ]:
# Pre-download NLTK corpora used downstream. The preprocessing module also
# downloads these on import, but doing it explicitly here surfaces any
# network issue before the pipeline starts.
import nltk
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

# 2. Analysing the Target Texts

Mirrors `run_text_analysis()` from `main.py`.

## 2.1 Pre-processing

Load, clean, and tokenize the two source texts (Chinese Room Argument and Minds, Brains, and Programs). The texts are fetched from Google Drive via `gdown` on first run. If a gdown rate-limit error appears, rerun this cell.

In [ ]:
from preprocessing import load_cra, load_mbp, count_text_stats

cra_raw, cra_clean, cra_tokens = load_cra()
mbp_raw, mbp_clean, mbp_tokens = load_mbp()

## 2.2 Exploratory Analysis

Count sentences, words, and unique words, then display word clouds of the most frequent terms in each text.

In [ ]:
from exploratory import print_text_stats, plot_wordclouds

cra_stats = count_text_stats(cra_raw)
mbp_stats = count_text_stats(mbp_raw)
print_text_stats("Chinese Room Argument (CRA)", *cra_stats)
print_text_stats("Minds, Brains, and Programs (MBP)", *mbp_stats)

plot_wordclouds(cra_tokens, mbp_tokens)

## 2.3 Topic Modelling

Use bag-of-words vectorization and Latent Dirichlet Allocation (LDA) to identify topics across both texts, and visualize each topic as a word cloud.

In [ ]:
from topic_modeling import run_lda, plot_topic_wordclouds

topics = run_lda([cra_clean, mbp_clean])
print("Identified topics:", topics)

plot_topic_wordclouds(topics)

## 2.4 Sentence Embedding

Use a Sentence-Transformer to embed each sentence, project the embeddings to 2D with t-SNE, and plot them with Plotly. Hover over points to inspect the underlying sentences.

*If the Plotly plot renders blank in Colab, run `import plotly.io as pio; pio.renderers.default = "colab"` and re-execute this cell.*

In [ ]:
from embedding import embed_sentences, reduce_to_2d, plot_embeddings

cra_sentences, cra_embeddings = embed_sentences(cra_raw)
mbp_sentences, mbp_embeddings = embed_sentences(mbp_raw)

x_cra, y_cra = reduce_to_2d(cra_embeddings)
x_mbp, y_mbp = reduce_to_2d(mbp_embeddings)

plot_embeddings(x_cra, y_cra, cra_sentences, x_mbp, y_mbp, mbp_sentences)

## 2.5 Text Summarization

Summarize both texts with [LongT5](https://huggingface.co/pszemraj/long-t5-tglobal-base-16384-book-summary), a text-to-text model fine-tuned for long-sequence summarization (Raffel et al., 2019; Guo et al., 2022).

The model (~1 GB) is downloaded on first use and runs on GPU automatically.

In [ ]:
from summarization import create_summarizer, summarize_and_display

summarizer = create_summarizer()
wrapped_cra_sum, wrapped_mbp_sum = summarize_and_display(summarizer, cra_raw, mbp_raw)

# 3. Applying Natural Language Inference to Argument Analysis

Natural Language Inference (NLI) is a classification task that takes a premise-hypothesis pair as input and predicts one of three inference types: entailment, contradiction, or neutral. The Stanford NLI corpus (SNLI) and the Multi-NLI corpus (MNLI) are commonly used as training datasets for NLI tasks.

## 3.1 Fine-tuning BERT on SNLI

> **⚠️ OPTIONAL — this cell takes approximately 30–50 minutes on a T4 GPU.**
>
> Section 3.2 uses a different, pre-trained CrossEncoder (DeBERTa-v3) and does **not** depend on the model trained here. Skip to 3.2 unless you specifically want to reproduce the BERT fine-tuning result (~0.80 accuracy on SNLI).
>
> If you plan to run the full pipeline top-to-bottom, consider `Runtime → Restart` before this cell to free VRAM held by LongT5 from Section 2.5.

Fine-tune BERT on SNLI following the methodology from [*Dive into Deep Learning* §16.7](https://d2l.ai/chapter_natural-language-processing-applications/natural-language-inference-bert.html).

In [ ]:
from bert_snli import load_pretrained_bert, get_devices, train_classifier

bert_model, _vocabulary = load_pretrained_bert()
devices = get_devices()
train_classifier(bert_model, devices)

## 3.2 Using an NLI Model for the Chinese Room Argument

Use a fine-tuned DeBERTa-v3 CrossEncoder (trained on SNLI + MNLI, 90.04% on MNLI topic-mismatched) to label the premise-conclusion relationships for four argument formulations: **MBP**, **CRA**, **LARG**, and **LARG_PARA**. See [Pre-trained Cross Encoders](https://www.sbert.net/docs/pretrained_cross-encoders.html#nli).

The combined DataFrame aggregates all four analyses; the bar chart compares model predictions to the expected (human) labels.

In [ ]:
from nli_analysis import load_nli_model, run_all_argument_analyses, plot_match_percentages

nli_model = load_nli_model()
combined_df = run_all_argument_analyses(nli_model)

print("\n--- Combined Results ---")
print(combined_df)

plot_match_percentages(combined_df)

# 4. Limits of Identifying the Argument Structure in Searle's Texts Using NLI

This section runs the full discussion pipeline (`discussion.run_discussion`), which covers:
- **4.1 Limits of Labeling Argument Types** — entailment-case subset of the combined DataFrame.
- **4.2 The Need for Argument Mining** — complexity of NLI over n-premise arguments compared with 2ⁿ combinations.
- **4.3 The Need for Probabilistic Analysis** — examples where a hard NLI label is insufficient.
- **4.4 The Need for Justification** — cases where the model's label is hard to justify without intermediate premises.

Running `run_discussion(...)` below emits the outputs for all four subsections (4.1 through 4.4) in a single call, reusing `nli_model`, `combined_df`, and the summarization/stats results from earlier cells.

In [ ]:
from discussion import run_discussion

run_discussion(
    nli_model,
    combined_df,
    cra_stats,
    mbp_stats,
    (wrapped_cra_sum, wrapped_mbp_sum),
)

# 5. Conclusion

I have analyzed texts from Searle (1980) and Searle (2001) using NLP techniques and applied NLI to specific arguments within these texts. The results and my subsequent analyses indicate that there are limitations to using NLI for identifying argumentative structures in such texts. I hope that future research will develop complementary techniques and justifications, thereby facilitating more accurate identification of argumentative structures in any given text.

# References

Bowman, S. R., Angeli, G., Potts, C., & Manning, C. D. (2015). A large annotated corpus for learning natural language inference. *arXiv preprint arXiv:1508.05326*.

Chen, T., Jiang, Z., Poliak, A., Sakaguchi, K., & Van Durme, B. (2019). Uncertain natural language inference. *arXiv preprint arXiv:1909.03042*.

Guo, M., Ainslie, J., Uthus, D., Ontañon, S., Ni, J., Sung, Y. H., & Yang, Y. (2022). LongT5: Efficient text-to-text transformer for long sequences. In *Findings of the Association for Computational Linguistics: NAACL 2022* (pp. 724–736).

Korman, D. Z., Mack, E., Jett, J., & Renear, A. H. (2018). Defining textual entailment. *Journal of the Association for Information Science and Technology*, 69(6), 763–772.

Raffel, C., Shazeer, N., Roberts, A., Lee, K., Narang, S., Matena, M., ... & Liu, P. J. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer. *The Journal of Machine Learning Research*, 21(1), 5485–5551.

Reimers, N., & Gurevych, I. (2019). Sentence-BERT: Sentence embeddings using Siamese BERT-networks. *arXiv preprint arXiv:1908.10084*.

Searle, J. R. (1980). Minds, brains, and programs. *Behavioral and Brain Sciences*, 3(3), 417–424.

Searle, J. R. (2001). Chinese room argument. In R. A. Wilson & F. C. Keil (Eds.), *The MIT Encyclopedia of the Cognitive Sciences*. MIT Press.

Williams, A., Nangia, N., & Bowman, S. R. (2017). A broad-coverage challenge corpus for sentence understanding through inference. *arXiv preprint arXiv:1704.05426*.

Zhang, A., Lipton, Z. C., Li, M., & Smola, A. J. (2023). *Dive into Deep Learning*.